# MedNorm-VI S1 - Full Mention Training

**FULL TRAINING. Colab Pro GPU runtime required. Long-running.**

This is the real S1 mention-extraction training run (spec §15: span BCE/focal + type loss on
the ViHealthBERT span+type head). It reuses the validated smoke components rather than
re-implementing them: the two-pass dependency bootstrap, the NumPy/AdamW ABI preflight, the
scoped S1 dependency-health gate, the governed corpus gate, VnCoreNLP production
segmentation, and the slow-tokenizer character alignment.

## TWO-PASS EXECUTION (required)

1. **PASS 1** - `Runtime > Run all`: installs the constrained dependency set, writes the
   bootstrap marker, and **restarts the kernel on purpose**.
2. **PASS 2** - `Runtime > Run all` **again**: the marker matches, installation is skipped,
   and training proceeds.

## EXPLICIT FULL-TRAINING GUARD

Training does not start unless `CONFIRM_FULL_TRAINING` is set to the exact phrase in the
tracked config. The default is empty, so an accidental `Run all` stops at the guard.

## PRECONDITIONS

* `MedNorm_S1_Smoke_Artifact_Validation.ipynb` passed on the real Drive artifact, and you
  supply the same `SMOKE_ARTIFACT_DIR` and `EXPECTED_SMOKE_CHECKPOINT_SHA256` here. The
  historical **v1** artifact (`s1_mention_first_run_smoke`) recorded
  `full_training_readiness: false` and cannot authorize training.
* `configs/training/s1_mention_full_training.yaml` has a **pinned 40-hex revision**.
  `main` is mutable and is rejected.

The one-step smoke checkpoint is execution evidence only. Full training initializes from the
approved pretrained ViHealthBERT revision and writes to a **separate** Drive directory.

## 1. Configuration and full-training guard (no scientific imports)

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = "https://github.com/vquclinh/MedNorm-VI.git"
REPO_REF = "main"

if "MEDNORM_DRIVE_ROOT" in os.environ:
    DRIVE_ROOT = Path(os.environ["MEDNORM_DRIVE_ROOT"])
if "MEDNORM_REPO_DIR" in os.environ:
    REPO_DIR = Path(os.environ["MEDNORM_REPO_DIR"])
if "MEDNORM_REPO_URL" in os.environ:
    REPO_URL = os.environ["MEDNORM_REPO_URL"]
if "MEDNORM_REPO_REF" in os.environ:
    REPO_REF = os.environ["MEDNORM_REPO_REF"]

CORPUS_DIR = DRIVE_ROOT / "data" / "derived" / "training_corpora" / "mednorm_vi_training_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"

# ---------------------------------------------------------------------------
# RUNTIME INPUTS - the smoke artifact that authorizes this run.
# ---------------------------------------------------------------------------
# Must be the artifact you validated with MedNorm_S1_Smoke_Artifact_Validation.
# v2 = corrected rerun; v1 (s1_mention_first_run_smoke) is historical evidence.
SMOKE_ARTIFACT_VERSION = os.environ.get("MEDNORM_SMOKE_ARTIFACT_VERSION", "v2")
SMOKE_ARTIFACT_DIR = Path(os.environ.get(
    "MEDNORM_SMOKE_ARTIFACT_DIR",
    str(DRIVE_ROOT / "artifacts" / f"s1_mention_first_run_smoke_{SMOKE_ARTIFACT_VERSION}")))
# 64-hex digest of the accepted smoke run. Empty blocks training (no source edit).
EXPECTED_SMOKE_CHECKPOINT_SHA256 = os.environ.get(
    "MEDNORM_EXPECTED_SMOKE_CHECKPOINT_SHA256", "")
HISTORICAL_SMOKE_ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke"
FULL_CONFIG_REL = Path("configs/training/s1_mention_full_training.yaml")

# ---------------------------------------------------------------------------
# FULL-TRAINING GUARD. Set this to the confirmation phrase in the tracked config
# to authorize a long GPU run. Empty means "stop before training".
# ---------------------------------------------------------------------------
CONFIRM_FULL_TRAINING = os.environ.get("MEDNORM_CONFIRM_FULL_TRAINING", "")

# Resume from the latest full-training checkpoint when one exists.
RESUME_FROM_LATEST = True

IN_COLAB_BOOTSTRAP = "google.colab" in sys.modules
print(json.dumps({
    "mode": "FULL_TRAINING",
    "drive_root": str(DRIVE_ROOT),
    "corpus_dir": str(CORPUS_DIR),
    "smoke_artifact_version": SMOKE_ARTIFACT_VERSION,
    "smoke_artifact_dir": str(SMOKE_ARTIFACT_DIR),
    "expected_checkpoint_sha256_supplied": bool(EXPECTED_SMOKE_CHECKPOINT_SHA256),
    "confirm_full_training_set": bool(CONFIRM_FULL_TRAINING),
    "resume_from_latest": RESUME_FROM_LATEST,
}, indent=2, sort_keys=True))

## 2. Repository checkout (stdlib only; no NumPy/Torch import)

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git",
    "clone",
    "--branch",
    REPO_REF,
    "--single-branch",
    REPO_URL,
    str(REPO_DIR),
], check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert len(RESOLVED_COMMIT) == 40 and all(c in "0123456789abcdef" for c in RESOLVED_COMMIT)
assert (REPO_DIR / "src" / "mednorm_vi").is_dir(), "cloned repo is missing src/mednorm_vi"
sys.path.insert(0, str(REPO_DIR / "src"))
print(json.dumps({"resolved_commit": RESOLVED_COMMIT, "src_verified": True}, indent=2, sort_keys=True))


## 3. Dependency metadata inspection (importlib.metadata; NumPy is NOT imported)

In [ ]:
import importlib.metadata as importlib_metadata

sys.path.insert(0, str(REPO_DIR / "src"))
from mednorm_vi.training.colab_bootstrap import (  # noqa: E402
    INSTALL_AND_RESTART,
    PROCEED,
    build_install_command,
    build_marker_fingerprint,
    build_pip_constraints,
    classify_dependency_health,
    compute_dependency_closure,
    decide_bootstrap_action,
    load_dependency_contract,
    marker_mismatches,
    normalize_distribution_name,
    validate_abi_report,
    validate_install_command,
)

DEPENDENCY_CONTRACT_PATH = REPO_DIR / "configs" / "training" / "s1_mention_colab_dependencies.yaml"
contract = load_dependency_contract(DEPENDENCY_CONTRACT_PATH)
DEPENDENCY_CONTRACT_VERSION = contract.contract_version

TRACKED_PACKAGES = (
    "numpy", "torch", "torchvision", "torchaudio", "transformers", "tokenizers",
    "huggingface_hub", "sentencepiece", "accelerate", "py_vncorenlp", "scipy",
    "pandas", "scikit-learn", "safetensors", "pyarrow",
)

def package_version(name):
    """Version via metadata only - never imports the package (no NumPy load)."""
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return ""

baseline_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
assert baseline_versions["numpy"], "no baseline NumPy detected in the Colab image"
assert baseline_versions["torch"], "no baseline Torch detected in the Colab image"

# The marker fingerprint binds a skipped installation to THIS tracked contract
# (exact file bytes), THIS Python major.minor, THIS protected baseline, and THIS
# normalized requirement set. A version string alone is far too weak.
PYTHON_MAJOR_MINOR = f"{sys.version_info.major}.{sys.version_info.minor}"
MARKER_FINGERPRINT = build_marker_fingerprint(
    contract, PYTHON_MAJOR_MINOR, baseline_versions)
DEPENDENCY_CONTRACT_SHA256 = contract.contract_sha256
INSTALL_REQUIREMENT_HASH = contract.install_requirement_hash
print(json.dumps({
    "baseline_versions": {k: v for k, v in baseline_versions.items() if v},
    "marker_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))


## 4. Consolidated installation + forced kernel restart (PASS 1 only)

In [ ]:
MARKER_PATH = Path(contract.marker_path)
CONSTRAINT_PATH = Path("/content/mednorm_s1_constraints.txt")

bootstrap_marker = None
if MARKER_PATH.is_file():
    try:
        bootstrap_marker = json.loads(MARKER_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        bootstrap_marker = None

# Every fingerprint field must match before installation may be skipped.
BOOTSTRAP_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
BOOTSTRAP_ACTION = decide_bootstrap_action(bootstrap_marker, MARKER_FINGERPRINT)
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_path": str(MARKER_PATH),
    "marker_mismatches": BOOTSTRAP_MARKER_MISMATCHES,
    "expected_fingerprint": MARKER_FINGERPRINT.as_dict(),
}, indent=2, sort_keys=True))

if BOOTSTRAP_ACTION == INSTALL_AND_RESTART:
    # Pin the INHERITED stack to the versions this runtime already provides so pip
    # cannot silently move NumPy/Torch. An incompatible request now fails loudly
    # instead of corrupting the C-ABI.
    constraints = build_pip_constraints(baseline_versions)
    CONSTRAINT_PATH.write_text("\n".join(constraints) + "\n", encoding="utf-8")
    install_command = build_install_command(contract, str(CONSTRAINT_PATH), sys.executable)
    validate_install_command(install_command)
    print("constraints:", constraints)
    print("install:", " ".join(install_command))
    subprocess.run(install_command, check=True)
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
    pip_check_output = "\n".join(
        part for part in (pip_check.stdout.strip(), pip_check.stderr.strip()) if part)
    # Written with the CURRENT fingerprint, so PASS 2 matches exactly and the
    # notebook can never restart twice for the same environment.
    MARKER_PATH.write_text(json.dumps({
        "install_completed": True,
        **MARKER_FINGERPRINT.as_dict(),
        "baseline_versions": baseline_versions,
        "constraints": constraints,
        "installed_specifiers": list(contract.install_specifiers),
        "pip_check_returncode": pip_check.returncode,
        "pip_check_stdout": pip_check.stdout,
        "pip_check_stderr": pip_check.stderr,
        "pip_check_output": pip_check_output,
    }, indent=2, sort_keys=True), encoding="utf-8")
    print("=" * 78)
    print("PASS 1 COMPLETE - the kernel is about to restart (this is expected).")
    print("AFTER the restart finishes, run Runtime > Run all AGAIN to execute PASS 2.")
    print("=" * 78)
    if IN_COLAB_BOOTSTRAP:
        os.kill(os.getpid(), 9)  # real kernel restart; Colab reconnects automatically
    else:
        raise SystemExit("dependency installation requires a kernel restart")
else:
    print("PASS 2: bootstrap marker matches every fingerprint field;",
          DEPENDENCY_CONTRACT_VERSION, DEPENDENCY_CONTRACT_SHA256[:16])

DEPENDENCY_RESTART_COMPLETED = BOOTSTRAP_ACTION == PROCEED


## 5. Post-restart dependency verification (scoped S1 health)

In [ ]:
assert DEPENDENCY_RESTART_COMPLETED, (
    "PASS 1 installs dependencies and restarts the kernel. Run all cells again for PASS 2.")

# Re-validate the marker in the restarted kernel: PASS 1 wrote it in a different
# process, so the fingerprint is re-checked here against the live runtime.
POST_RESTART_MARKER_MISMATCHES = marker_mismatches(bootstrap_marker, MARKER_FINGERPRINT)
assert not POST_RESTART_MARKER_MISMATCHES, (
    f"bootstrap marker no longer matches this runtime: {POST_RESTART_MARKER_MISMATCHES}")

installed_versions = {name: package_version(name) for name in TRACKED_PACKAGES}
marker_baseline = dict(bootstrap_marker.get("baseline_versions", {}))
changed_packages = {
    name: {"baseline": marker_baseline.get(name, ""), "current": installed_versions[name]}
    for name in TRACKED_PACKAGES
    if marker_baseline.get(name, "") != installed_versions[name]
}
protected_changed = {
    name: change for name, change in changed_packages.items()
    if name in ("numpy", "torch", "torchvision", "torchaudio")
}
assert not protected_changed, (
    f"inherited stack was modified despite constraints: {protected_changed}")
# `pip check` is captured IN FULL (stdout and stderr, never truncated) and kept as a
# diagnostic. It audits the whole Colab image, so its global verdict alone must not
# gate S1: preinstalled Gradio/IPython complaints are unrelated to this smoke.
pip_check_proc = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
PIP_CHECK_OUTPUT = "\n".join(
    part for part in (pip_check_proc.stdout.strip(), pip_check_proc.stderr.strip()) if part)
PIP_CHECK_PASSED = pip_check_proc.returncode == 0

def installed_requirement_graph():
    """Requirement graph from metadata only - imports nothing (no NumPy load)."""
    graph = {}
    for dist in importlib_metadata.distributions():
        dist_name = normalize_distribution_name(dist.metadata["Name"] or "")
        if dist_name:
            graph.setdefault(dist_name, []).extend(dist.requires or [])
    return graph

# The closure is what S1 ACTUALLY depends on: the contract's import roots plus their
# transitive requirements, resolved from the installed metadata of this runtime.
S1_DEPENDENCY_CLOSURE = compute_dependency_closure(
    contract.closure_root_distributions, installed_requirement_graph())
DEPENDENCY_HEALTH = classify_dependency_health(
    PIP_CHECK_OUTPUT, S1_DEPENDENCY_CLOSURE, pip_check_proc.returncode)
print("pip check output (complete):")
print(PIP_CHECK_OUTPUT or "(no broken requirements reported)")
print(json.dumps({
    "bootstrap_action": BOOTSTRAP_ACTION,
    "marker_mismatches": POST_RESTART_MARKER_MISMATCHES,
    "dependency_contract_sha256": DEPENDENCY_CONTRACT_SHA256,
    "install_requirement_hash": INSTALL_REQUIREMENT_HASH,
    "python_major_minor": PYTHON_MAJOR_MINOR,
    "changed_packages": changed_packages,
    "protected_stack_unchanged": True,
    "pip_check_passed": PIP_CHECK_PASSED,
    "s1_dependency_closure_size": len(S1_DEPENDENCY_CLOSURE),
    "s1_dependency_healthy": DEPENDENCY_HEALTH.healthy,
    "blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.blocking],
    "non_blocking_dependency_conflicts": [c.message for c in DEPENDENCY_HEALTH.non_blocking],
}, indent=2, sort_keys=True))
if DEPENDENCY_HEALTH.non_blocking:
    print("NOTE: the conflicts above are outside the S1 dependency closure and are")
    print("      recorded as non-blocking diagnostics. They are NOT remediated here:")
    print("      huggingface_hub is not upgraded for Gradio, and NumPy/Torch are untouched.")


## 6. NumPy / AdamW ABI preflight (fail fast before any acquisition)

In [ ]:
# FAIL-FAST NumPy/Torch C-ABI health. This is the FIRST place NumPy or Torch is
# imported, and it runs BEFORE any Drive mount, corpus, VnCoreNLP, tokenizer, or
# model acquisition. The Audit 0022 run died here in disguise: the ABI was already
# broken, and torch.optim.AdamW merely triggered the first compiled numpy.random import.
abi_report = {
    "numpy_imported": False,
    "numpy_random_imported": False,
    "torch_imported": False,
    "adamw_constructed": False,
    "pip_check_passed": PIP_CHECK_PASSED,
}
try:
    import numpy as np  # noqa: E402
    abi_report["numpy_imported"] = True

    from numpy.random import RandomState  # noqa: E402

    rng = RandomState(42)
    values = rng.rand(4)
    assert values.shape == (4,)
    abi_report["numpy_random_imported"] = True

    import numpy.random.mtrand as numpy_mtrand  # noqa: E402

    abi_report.update({
        "numpy_version": np.__version__,
        "numpy_path": str(Path(np.__file__).resolve().parent),
        "numpy_mtrand_path": str(Path(numpy_mtrand.__file__).resolve()),
    })
    # numpy.core is a DEPRECATED NumPy 2.x compatibility shim; it is a diagnostic
    # path only, so its absence must never be read as an ABI failure.
    try:
        import numpy.core as numpy_core  # noqa: E402

        abi_report["numpy_core_path"] = str(Path(numpy_core.__file__).resolve().parent)
    except Exception as core_exc:  # noqa: BLE001 - diagnostic only
        abi_report["numpy_core_path"] = f"unavailable: {type(core_exc).__name__}"

    import torch  # noqa: E402

    abi_report["torch_imported"] = True
    abi_report.update({
        "torch_version": torch.__version__,
        "torch_path": str(Path(torch.__file__).resolve().parent),
        "cuda_available": bool(torch.cuda.is_available()),
    })

    dummy_parameter = torch.nn.Parameter(torch.zeros(1))
    dummy_optimizer = torch.optim.AdamW([dummy_parameter], lr=1e-3)
    dummy_optimizer.zero_grad(set_to_none=True)
    abi_report["adamw_constructed"] = True

    # Validate the ACTUAL S1 dependency closure: every module S1 imports must load.
    # This is the positive check that replaces a global pip check verdict.
    s1_import_failures = []
    s1_import_versions = {}
    for _distribution, _module in contract.import_closure_roots:
        try:
            _imported = importlib.import_module(_module)
            s1_import_versions[_module] = str(
                getattr(_imported, "__version__", "") or package_version(_distribution))
        except Exception as import_exc:  # noqa: BLE001 - collect every failure
            s1_import_failures.append(f"{_module}: {type(import_exc).__name__}: {import_exc}")
    abi_report["s1_import_failures"] = s1_import_failures
    abi_report["s1_import_versions"] = s1_import_versions
    abi_report["s1_dependency_closure_verified"] = True
except Exception as exc:  # noqa: BLE001 - diagnostics then fail fast
    abi_report["error"] = f"{type(exc).__name__}: {exc}"
    print(json.dumps({
        "abi_preflight": "FAILED",
        "report": abi_report,
        "python": platform.python_version(),
        "sys_path_head": sys.path[:5],
        "pythonpath": os.environ.get("PYTHONPATH", ""),
    }, indent=2, sort_keys=True))
    raise

abi_report["numpy_distribution_count"] = sum(
    1 for dist in importlib_metadata.distributions()
    if (dist.metadata["Name"] or "").lower() == "numpy")
abi_report["python_version"] = platform.python_version()
# Closure-scoped dependency health. `pip_check_output` is carried in full; only
# conflicts raised BY the S1 closure can produce a blocking problem.
abi_report.update(DEPENDENCY_HEALTH.as_dict())

abi_problems = validate_abi_report(abi_report)
assert not abi_problems, f"NumPy/Torch ABI preflight failed: {abi_problems}"
NUMPY_ABI_PREFLIGHT_PASSED = True
S1_DEPENDENCY_CLOSURE_VERIFIED = True
print(json.dumps({"abi_preflight": "PASSED", "report": abi_report}, indent=2, sort_keys=True))


## 7. Runtime, GPU, and Drive mount

In [ ]:
runtime_report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB_BOOTSTRAP,
    "dependency_contract_version": DEPENDENCY_CONTRACT_VERSION,
    "dependency_restart_completed": DEPENDENCY_RESTART_COMPLETED,
    "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
    "s1_dependency_closure_verified": S1_DEPENDENCY_CLOSURE_VERIFIED,
}
assert IN_COLAB_BOOTSTRAP, "S1 full training must run on Google Colab Pro with a GPU."
assert torch.cuda.is_available(), "S1 full training requires a Colab GPU runtime."
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
SUPPORTS_BF16 = bool(torch.cuda.is_bf16_supported())
runtime_report.update({
    "torch": torch.__version__,
    "cuda_available": True,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_vram_gb": round(props.total_memory / 1e9, 2),
    "supports_bf16": SUPPORTS_BF16,
})

drive_module = importlib.import_module("google.colab.drive")
drive_module.mount("/content/drive")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(runtime_report, indent=2, sort_keys=True))

## 8. Smoke-artifact gate and immutable revision pin

Full training is authorized only by a **validated** smoke artifact. The checkpoint SHA-256 is
recomputed from the bytes on Drive; the immutable model revision is read from that validated
manifest and never invented.

In [ ]:
from mednorm_vi.training.s1_artifact_validation import (  # noqa: E402
    pinned_revision_from_outcome,
    validate_smoke_artifact,
)
from mednorm_vi.training.s1_full_training import (  # noqa: E402
    BEST_METRIC_KEY,
    FULL_TRAINING_MODE,
    MentionMetrics,
    build_checkpoint_payload,
    build_full_training_manifest,
    derive_schedule,
    full_training_output_paths,
    is_better_metric,
    is_supervised_example,
    load_full_training_config,
    validate_resume_checkpoint,
)
from mednorm_vi.training.s1_mention_smoke import load_smoke_config  # noqa: E402

smoke_config = load_smoke_config(
    REPO_DIR / "configs" / "training" / "s1_mention_first_run_smoke.yaml")
assert SMOKE_ARTIFACT_DIR.resolve() != HISTORICAL_SMOKE_ARTIFACT_DIR.resolve(), (
    "the historical v1 smoke artifact recorded full_training_readiness: false and "
    "must not authorize full training; validate and use the corrected rerun")
smoke_outcome = validate_smoke_artifact(
    SMOKE_ARTIFACT_DIR,
    expected_corpus=smoke_config["corpus"],
    expected_checkpoint_sha256=EXPECTED_SMOKE_CHECKPOINT_SHA256,
)
if not smoke_outcome.smoke_validated:
    for failure in smoke_outcome.failures:
        print("  -", failure)
    if not smoke_outcome.diagnostics["expected_checkpoint_sha256_supplied"]:
        print(f"Recomputed checkpoint SHA-256: {smoke_outcome.checkpoint_sha256}")
        print("Set EXPECTED_SMOKE_CHECKPOINT_SHA256 in cell 1 to accept that run.")
    raise AssertionError(
        f"smoke artifact validation failed ({len(smoke_outcome.failures)} condition(s)); "
        "full training is not authorized")

# Blocker, not a guess: full training stops if no immutable revision is recorded.
PINNED_MODEL_REVISION = pinned_revision_from_outcome(smoke_outcome)
# Full training consumes the artifact that was just validated, not a hardcoded path.
config = load_full_training_config(
    REPO_DIR / FULL_CONFIG_REL, pinned_revision=PINNED_MODEL_REVISION,
    smoke_artifact_dir=SMOKE_ARTIFACT_DIR)
OUTPUT_DIR = Path(config.output_dir)
assert OUTPUT_DIR.resolve() != SMOKE_ARTIFACT_DIR.resolve(), (
    "full-training output must never overwrite the validated smoke artifact")
OUTPUT_PATHS = full_training_output_paths(OUTPUT_DIR)
for key in ("latest_checkpoint", "best_checkpoint", "training_history"):
    Path(OUTPUT_PATHS[key]).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_PATHS["resolved_config"]).write_text(
    json.dumps(config.resolved(), indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({
    "smoke_artifact_validated": True,
    "validated_smoke_artifact_dir": str(SMOKE_ARTIFACT_DIR),
    "smoke_checkpoint_sha256": smoke_outcome.checkpoint_sha256,
    "pinned_model_revision": PINNED_MODEL_REVISION,
    "output_dir": str(OUTPUT_DIR),
    "effective_batch_size": config.effective_batch_size,
    "config_sha256": config.config_sha256,
}, indent=2, sort_keys=True))

## 9. Full-training guard (training stops here without explicit confirmation)

In [ ]:
EXPECTED_PHRASE = str(config.raw["runtime"]["confirmation_phrase"])
if CONFIRM_FULL_TRAINING != EXPECTED_PHRASE:
    print("=" * 78)
    print("FULL TRAINING NOT CONFIRMED - stopping before any long GPU run.")
    print(f'Set CONFIRM_FULL_TRAINING = "{EXPECTED_PHRASE}" in cell 1 and re-run.')
    print("=" * 78)
    raise SystemExit("full training requires explicit confirmation")
FULL_TRAINING_CONFIRMED = True
print("full training CONFIRMED:", EXPECTED_PHRASE)

## 10. Governed corpus gate (identical to the smoke gate)

In [ ]:
from mednorm_vi.model_registry.registry import load_registry, validate_profile_budget
from mednorm_vi.training.phobert_alignment import (
    ALIGNMENT_BACKEND,
    BOUNDARY_MERGE_POLICY,
    STAGE_SUBTOKEN_ENCODING,
    PARTIAL_TRUNCATION_POLICY,
    SUBTOKEN_SUPERVISION_POLICY,
    AlignmentError,
    describe_backend,
    map_segmented_words,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)
from mednorm_vi.training.s1_mention_smoke import (
    ENTITY_TYPE_ORDER,
    alignment_diagnostic,
    encode_mention_example_slow,
    governed_exclusion_diagnostic,
    load_governed_exclusions,
    privacy_safe_example_id,
    summarize_alignment_diagnostics,
    expected_corpus_from_config,
    iter_jsonl,
    load_coverage,
    loss_mask_for_example,
    pad_encoded_features,
    sha256_file,
    verify_governed_corpus,
)

expected_corpus = expected_corpus_from_config(config.raw)
corpus_report = verify_governed_corpus(CORPUS_DIR, expected_corpus)
coverage_by_source = load_coverage(CORPUS_DIR)

roles = load_registry(REPO_DIR / "configs" / "model_registry" / "models_v1.yaml")
role = next(r for r in roles if r.model_id == config.registry_model_id)
profile_budget = validate_profile_budget(roles, profile="full")
assert profile_budget.within_9b, "full model profile exceeds the 9B budget"
print(json.dumps({**corpus_report, "registry_role": role.role,
                  "full_profile_within_9b": profile_budget.within_9b},
                 indent=2, sort_keys=True))

## 11. Word segmentation contract (VnCoreNLP RDRSegmenter required)

In [ ]:
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"
# Production S1 training REQUIRES VnCoreNLP RDRSegmenter. whitespace_fallback is an
# explicit opt-in diagnostic mode only; it never activates automatically.
SEGMENTER_MODE = os.environ.get("MEDNORM_SEGMENTER_MODE", "vncorenlp")
assert SEGMENTER_MODE in ("vncorenlp", "whitespace_fallback"), SEGMENTER_MODE
DEGRADED_FALLBACK = SEGMENTER_MODE == "whitespace_fallback"

segmenter_report = {
    "segmenter_mode": SEGMENTER_MODE,
    "word_segmenter": "",
    "word_segmenter_version": "",
    "word_segmenter_resource_hashes": {},
    "degraded_fallback": DEGRADED_FALLBACK,
    "resource_dir": str(VNCORENLP_DIR),
    "acquisition_source": "",
}

if SEGMENTER_MODE == "vncorenlp":
    # py_vncorenlp was installed in the single constrained transaction (PASS 1).
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    py_vncorenlp = importlib.import_module("py_vncorenlp")
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _resources = sorted(
        [q for q in VNCORENLP_DIR.rglob("*") if q.is_file()], key=lambda q: q.name)
    assert _resources, "VnCoreNLP resources missing after acquisition (fail fast)"
    _hashes = {q.name: sha256_file(q) for q in _resources if q.suffix in (".jar", ".xz", ".txt")}
    assert _hashes, "VnCoreNLP resource hashes are empty (broken installation; fail fast)"
    _rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    segmenter_report.update({
        "word_segmenter": "VnCoreNLP RDRSegmenter",
        "word_segmenter_version": "py_vncorenlp==0.1.4",
        "word_segmenter_resource_hashes": _hashes,
        "acquisition_source": "py_vncorenlp.download_model",
    })

    def segment_example_text(text):
        segments = _rdr.word_segment(text)
        assert segments, "RDRSegmenter returned no segments (fail fast)"
        return " ".join(segments)
else:
    print("=" * 78)
    print("!! DEGRADED MODE: whitespace_fallback is NOT the production S1 path.")
    print("!! Word segmentation does not match ViHealthBERT-Word pretraining.")
    print("!! This run cannot be classified as a successful production-path S1 smoke.")
    print("=" * 78)
    segmenter_report.update({
        "word_segmenter": "whitespace (degraded diagnostics only)",
        "word_segmenter_version": "builtin-whitespace",
        "acquisition_source": "none (degraded diagnostics mode)",
    })

    def segment_example_text(text):
        return " ".join(text.split())

PRODUCTION_SEGMENTATION = (
    segmenter_report["segmenter_mode"] == "vncorenlp"
    and segmenter_report["word_segmenter"] == "VnCoreNLP RDRSegmenter"
    and segmenter_report["degraded_fallback"] is False
    and bool(segmenter_report["word_segmenter_resource_hashes"])
)
print(json.dumps({k: v for k, v in segmenter_report.items()
                  if k != "word_segmenter_resource_hashes"}, indent=2, sort_keys=True))
print("resource_hash_count", len(segmenter_report["word_segmenter_resource_hashes"]))
print("production_segmentation", PRODUCTION_SEGMENTATION)


In [ ]:
assert PRODUCTION_SEGMENTATION, (
    "full S1 training requires production VnCoreNLP segmentation; "
    "whitespace_fallback is diagnostics only")

## 12. Slow tokenizer at the PINNED immutable revision

In [ ]:
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

transformers_module = importlib.import_module("transformers")
AutoModel = transformers_module.AutoModel
AutoTokenizer = transformers_module.AutoTokenizer
runtime_report["transformers"] = transformers_module.__version__

# ViHealthBERT-Word declares tokenizer_class = PhobertTokenizer, which has NO fast
# implementation. Load it honestly as slow and align characters manually.
tokenizer = AutoTokenizer.from_pretrained(
    config.hf_model_id,
    revision=PINNED_MODEL_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=False,
)
tokenizer_report = describe_backend(tokenizer)
tokenizer_report.update({
    "tokenizer_revision": PINNED_MODEL_REVISION,
    "pad_token_id": tokenizer.pad_token_id,
})
assert tokenizer_report["tokenizer_is_fast"] is False, (
    "expected the slow PhobertTokenizer; a fast tokenizer would change the alignment contract")
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1
print(json.dumps(tokenizer_report, indent=2, sort_keys=True))

## 13. Encode the governed splits

Only examples whose governed loss mask enables span **and** entity_type supervision are used.
`phoner_covid19` declares `boundary: false`, so its `label_mask` is entirely zero and it would
contribute no gradient at all; sampling is scoped, and the governed corpus files are untouched.

Tokenizer equivalence is verified on a deterministic sample before encoding, exactly as the
smoke did. Encoding is not cached: re-encoding on resume is a few minutes and avoids storing a
large derived artifact on Drive.

In [ ]:
GOVERNED_EXCLUSIONS = load_governed_exclusions(
    REPO_DIR / "configs" / "training" / "s1_governed_exclusions.yaml")


def encode_split(split_name, *, limit=None):
    """Encode one governed split with production segmentation and slow alignment.

    Failures are recorded as PRIVACY-SAFE diagnostics (dataset, split, hashed
    example id, stage, reason code, exception class) - never raw text. Unexpected
    failures are counted separately from tracked governed exclusions.
    """
    path = CORPUS_DIR / "splits" / f"{split_name}.jsonl"
    features, skipped_unsupervised, considered = [], 0, 0
    diagnostics = []
    truncated_entities = partially_truncated = fully_dropped = 0
    merged_words = merged_entities = 0
    started = time.time()
    for index, row in enumerate(iter_jsonl(path)):
        if limit is not None and len(features) >= limit:
            break
        if config.filter_unsupervised_examples and not is_supervised_example(
                loss_mask_for_example(row, coverage_by_source)):
            skipped_unsupervised += 1
            continue
        considered += 1
        if privacy_safe_example_id(row["example_id"]) in GOVERNED_EXCLUSIONS:
            diagnostics.append(governed_exclusion_diagnostic(row, split=split_name))
            continue
        try:
            feature = encode_mention_example_slow(
                row, tokenizer, coverage_by_source=coverage_by_source,
                max_length=config.max_sequence_length,
                segmented_text=segment_example_text(row["text"]),
            )
        except (AlignmentError, ValueError) as exc:
            diagnostics.append(alignment_diagnostic(
                row, split=split_name, stage=STAGE_SUBTOKEN_ENCODING, error=exc))
            continue
        truncated_entities += int(feature["truncated_entity_count"])
        partially_truncated += int(feature["partially_truncated_entity_count"])
        fully_dropped += int(feature["fully_dropped_entity_count"])
        merged_words += int(feature["boundary_merge_masked_word_count"])
        merged_entities += int(feature["boundary_merge_affected_entity_count"])
        features.append(feature)
        if index % 2000 == 0:
            print(f"  {split_name}: {index} rows -> {len(features)} features "
                  f"({time.time() - started:.0f}s)", flush=True)
    summary = summarize_alignment_diagnostics(diagnostics)
    report = {
        "split": split_name,
        "examples_considered": considered,
        "encoded_examples": len(features),
        "skipped_unsupervised_examples": skipped_unsupervised,
        "truncated_entity_count": truncated_entities,
        "partially_truncated_entity_count": partially_truncated,
        "fully_dropped_entity_count": fully_dropped,
        "boundary_merge_masked_word_count": merged_words,
        "boundary_merge_affected_entity_count": merged_entities,
        "boundary_merge_policy": BOUNDARY_MERGE_POLICY,
        "encode_seconds": round(time.time() - started, 1),
        **summary,
    }
    report["counters_reconciled"] = (
        len(features) + summary["unalignable_example_count"]
        + summary["governed_exclusion_count"]) == considered
    assert report["counters_reconciled"], f"{split_name} counters do not reconcile: {report}"
    return features, report

# Tokenizer equivalence preflight on a deterministic sample (fail fast, before encoding).
tokenizer_equivalence = {"tokenizer_equivalence_checked": True,
                         "tokenizer_equivalence_examples": 0,
                         "tokenizer_equivalence_failures": 0}
for row in list(iter_jsonl(CORPUS_DIR / "splits" / "validation.jsonl"))[:64]:
    try:
        words = map_segmented_words(
            row["text"], segmented_text_to_words(segment_example_text(row["text"])))
        verify_tokenizer_equivalence(words, tokenizer)
        tokenizer_equivalence["tokenizer_equivalence_examples"] += 1
    except AlignmentError:
        tokenizer_equivalence["tokenizer_equivalence_failures"] += 1
assert tokenizer_equivalence["tokenizer_equivalence_failures"] == 0, (
    "per-word alignment does not match whole-sentence tokenization")
assert tokenizer_equivalence["tokenizer_equivalence_examples"] > 0

train_features, train_encoding_report = encode_split("train")
validation_features, validation_encoding_report = encode_split("validation")
assert train_features and validation_features, "encoding produced no usable features"
# Unexpected alignment failures block full training; governed exclusions do not.
for _report in (train_encoding_report, validation_encoding_report):
    if _report["unalignable_examples"]:
        print(f"UNEXPECTED unalignable examples in {_report['split']}:")
        for _entry in _report["unalignable_examples"][:20]:
            print("  ", json.dumps(_entry, sort_keys=True))
    assert _report["unalignable_example_count"] == 0, (
        f"{_report['split']}: {_report['unalignable_example_count']} unexpected "
        "unalignable example(s); fix the alignment backend or record an explicit "
        "governed exclusion - never ignore them")
print(json.dumps({"train": train_encoding_report,
                  "validation": validation_encoding_report,
                  **tokenizer_equivalence}, indent=2, sort_keys=True))

## 14. Deterministic DataLoaders

In [ ]:
import numpy as np  # noqa: E402
import random  # noqa: E402

SEED = config.seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


class MentionDataset(torch.utils.data.Dataset):
    def __init__(self, features):
        self.features = list(features)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]


def collate(batch):
    padded = pad_encoded_features(batch, pad_token_id=PAD_TOKEN_ID)
    return {
        "input_ids": torch.tensor(padded["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(padded["attention_mask"], dtype=torch.long),
        "labels": torch.tensor(padded["labels"], dtype=torch.float32),
        "label_mask": torch.tensor(padded["label_mask"], dtype=torch.float32),
    }


generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = torch.utils.data.DataLoader(
    MentionDataset(train_features), batch_size=config.per_device_batch_size,
    shuffle=True, collate_fn=collate, generator=generator, drop_last=False, num_workers=0)
validation_loader = torch.utils.data.DataLoader(
    MentionDataset(validation_features), batch_size=config.per_device_batch_size,
    shuffle=False, collate_fn=collate, num_workers=0)

schedule = derive_schedule(config, len(train_features))
print(json.dumps({**schedule.as_dict(),
                  "train_batches_per_epoch": len(train_loader),
                  "validation_batches": len(validation_loader)},
                 indent=2, sort_keys=True))

## 15. Model, optimizer, scheduler

The backbone is loaded from the **pinned immutable revision** of the approved pretrained
ViHealthBERT model - never from the smoke checkpoint. Weight decay is not applied to bias or
LayerNorm parameters, and the randomly initialized head gets a higher learning rate.

In [ ]:
class MentionTokenClassifier(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, label_count: int) -> None:
        super().__init__()
        self.base_model = base_model
        hidden_size = int(base_model.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(hidden_size, label_count)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(output.last_hidden_state))


backbone = AutoModel.from_pretrained(
    config.hf_model_id, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR))
resolved_model_revision = getattr(backbone.config, "_commit_hash", "") or PINNED_MODEL_REVISION
assert resolved_model_revision == PINNED_MODEL_REVISION, (
    f"resolved revision {resolved_model_revision} != pinned {PINNED_MODEL_REVISION}")
model = MentionTokenClassifier(backbone, len(ENTITY_TYPE_ORDER)).to(device)

NO_DECAY = ("bias", "LayerNorm.weight", "layer_norm.weight")
parameter_groups = [
    {"params": [p for n, p in model.base_model.named_parameters()
                if not any(k in n for k in NO_DECAY)],
     "lr": config.learning_rate, "weight_decay": config.weight_decay},
    {"params": [p for n, p in model.base_model.named_parameters()
                if any(k in n for k in NO_DECAY)],
     "lr": config.learning_rate, "weight_decay": 0.0},
    {"params": list(model.classifier.parameters()),
     "lr": config.head_learning_rate, "weight_decay": config.weight_decay},
]
optimizer = torch.optim.AdamW(parameter_groups)
scheduler = transformers_module.get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=schedule.warmup_steps,
    num_training_steps=schedule.total_optimizer_steps)

AMP_DTYPE = None
if config.mixed_precision in ("auto", "bf16") and SUPPORTS_BF16:
    AMP_DTYPE = torch.bfloat16
elif config.mixed_precision in ("auto", "fp16"):
    AMP_DTYPE = torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

trainable_parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(json.dumps({
    "pinned_model_revision": PINNED_MODEL_REVISION,
    "resolved_model_revision": resolved_model_revision,
    "base_parameters": sum(p.numel() for p in backbone.parameters()),
    "trainable_parameters": trainable_parameter_count,
    "amp_dtype": str(AMP_DTYPE), "warmup_steps": schedule.warmup_steps,
}, indent=2, sort_keys=True))

## 16. Loss, evaluation, and resume

In [ ]:
def masked_mention_loss(logits, labels, label_mask):
    """Spec §15: span BCE/focal + type loss, masked to supervised tokens."""
    per_element = torch.nn.functional.binary_cross_entropy_with_logits(
        logits, labels, reduction="none")
    if config.loss_type == "focal":
        probability = torch.sigmoid(logits)
        p_t = probability * labels + (1.0 - probability) * (1.0 - labels)
        alpha_t = config.focal_alpha * labels + (1.0 - config.focal_alpha) * (1.0 - labels)
        per_element = alpha_t * (1.0 - p_t).pow(config.focal_gamma) * per_element
    mask = label_mask.unsqueeze(-1)
    normalizer = torch.clamp(label_mask.sum() * len(ENTITY_TYPE_ORDER), min=1.0)
    return (per_element * mask).sum() / normalizer


@torch.no_grad()
def evaluate():
    model.eval()
    metrics = MentionMetrics()
    losses = []
    for batch in validation_loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss = masked_mention_loss(logits, batch["labels"], batch["label_mask"])
        losses.append(float(loss.detach().float().cpu()))
        predictions = (torch.sigmoid(logits.float()) > config.decision_threshold).int()
        metrics.update(predictions.tolist(), batch["labels"].int().tolist(),
                       batch["label_mask"].int().tolist())
    model.train()
    return {"validation_loss": sum(losses) / max(1, len(losses)), **metrics.compute()}


start_epoch, global_step, best_metric = 0, 0, None
latest_path = Path(OUTPUT_PATHS["latest_checkpoint"])
if RESUME_FROM_LATEST and latest_path.is_file():
    payload = torch.load(latest_path, map_location="cpu", weights_only=False)
    problems = validate_resume_checkpoint(payload, config)
    assert not problems, f"cannot resume from {latest_path}: {problems}"
    model.load_state_dict(payload["model_state_dict"])
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    scheduler.load_state_dict(payload["scheduler_state_dict"])
    start_epoch = int(payload["epoch"])
    global_step = int(payload["global_step"])
    best_metric = payload["best_metric"]
    print(f"RESUMED from {latest_path}: epoch={start_epoch} step={global_step} "
          f"best={best_metric}")
else:
    print("starting a fresh full-training run (no resumable latest checkpoint)")

## 17. Training loop

Bounded gradient handling, gradient accumulation, mixed precision, periodic progress logging,
per-epoch validation, `latest` + `best` checkpointing, and explicit OOM guidance.

In [ ]:
history_path = Path(OUTPUT_PATHS["training_history"])
run_completed = False
interrupted_reason = ""
validation_metrics = {}
completed_epochs = start_epoch
LOG_EVERY_STEPS = 50


def save_checkpoint(destination, epoch, step, best):
    payload = build_checkpoint_payload(
        config, epoch=epoch, global_step=step, best_metric=float(best if best is not None else 0.0),
        model_state_dict=model.state_dict(),
        optimizer_state_dict=optimizer.state_dict(),
        scheduler_state_dict=scheduler.state_dict())
    torch.save(payload, destination)
    return sha256_file(destination)


def log_history(record):
    with history_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, sort_keys=True) + "\n")


try:
    model.train()
    for epoch in range(start_epoch, config.num_epochs):
        epoch_started = time.time()
        running_loss, micro_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                logits = model(batch["input_ids"], batch["attention_mask"])
                loss = masked_mention_loss(logits, batch["labels"], batch["label_mask"])
            assert torch.isfinite(loss).item(), f"non-finite loss at step {global_step}"
            scaled = loss / config.gradient_accumulation_steps
            if scaler.is_enabled():
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            running_loss += float(loss.detach().float().cpu())
            micro_batches += 1
            if (step + 1) % config.gradient_accumulation_steps == 0 or (
                    step + 1) == len(train_loader):
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % LOG_EVERY_STEPS == 0:
                    record = {
                        "event": "train", "epoch": epoch, "global_step": global_step,
                        "train_loss": running_loss / max(1, micro_batches),
                        "learning_rate": scheduler.get_last_lr()[0],
                        "seconds": round(time.time() - epoch_started, 1),
                    }
                    log_history(record)
                    print(json.dumps(record, sort_keys=True), flush=True)
                    running_loss, micro_batches = 0.0, 0

        validation_metrics = evaluate()
        completed_epochs = epoch + 1
        latest_sha = save_checkpoint(
            latest_path, completed_epochs, global_step, best_metric)
        candidate = float(validation_metrics[BEST_METRIC_KEY])
        improved = is_better_metric(candidate, best_metric)
        if improved:
            best_metric = candidate
            save_checkpoint(Path(OUTPUT_PATHS["best_checkpoint"]),
                            completed_epochs, global_step, best_metric)
        record = {"event": "validation", "epoch": completed_epochs,
                  "global_step": global_step, "best_metric_improved": improved,
                  "best_metric": best_metric, "latest_checkpoint_sha256": latest_sha,
                  "epoch_seconds": round(time.time() - epoch_started, 1),
                  **validation_metrics}
        log_history(record)
        print(json.dumps(record, indent=2, sort_keys=True), flush=True)
    run_completed = True
except torch.cuda.OutOfMemoryError as exc:
    interrupted_reason = f"CUDA out of memory: {exc}"
    torch.cuda.empty_cache()
    print("=" * 78)
    print("CUDA OUT OF MEMORY. The latest checkpoint on Drive is still resumable.")
    print("Recovery options, in order of preference:")
    print("  1. halve optimization.per_device_batch_size and double")
    print("     gradient_accumulation_steps (the effective batch size is unchanged);")
    print("  2. lower data.max_sequence_length (256 -> 192);")
    print("  3. request a larger Colab GPU and re-run - training resumes automatically.")
    print("=" * 78)
    raise
except KeyboardInterrupt:
    interrupted_reason = "interrupted by user"
    print("interrupted; the latest checkpoint on Drive remains resumable")

## 18. Full-training manifest

In [ ]:
checkpoint_hashes = {}
for key in ("latest_checkpoint", "best_checkpoint"):
    path = Path(OUTPUT_PATHS[key])
    if path.is_file():
        checkpoint_hashes[key] = sha256_file(path)

Path(OUTPUT_PATHS["validation_metrics"]).write_text(
    json.dumps(validation_metrics, indent=2, sort_keys=True), encoding="utf-8")

training_manifest = build_full_training_manifest(
    config,
    schedule=schedule,
    repository={"repo_url": REPO_URL, "repo_ref": REPO_REF,
                "resolved_commit": RESOLVED_COMMIT},
    corpus=corpus_report,
    environment={
        "dependency_contract_version": DEPENDENCY_CONTRACT_VERSION,
        "dependency_contract_sha256": DEPENDENCY_CONTRACT_SHA256,
        "install_requirement_hash": INSTALL_REQUIREMENT_HASH,
        "python_major_minor": PYTHON_MAJOR_MINOR,
        "dependency_restart_completed": DEPENDENCY_RESTART_COMPLETED,
        "numpy_abi_preflight_passed": NUMPY_ABI_PREFLIGHT_PASSED,
        "s1_dependency_closure_verified": S1_DEPENDENCY_CLOSURE_VERIFIED,
        "runtime": runtime_report,
        **DEPENDENCY_HEALTH.as_dict(),
    },
    segmentation=segmenter_report,
    alignment={
        "alignment_backend": ALIGNMENT_BACKEND,
        "subtoken_supervision_policy": SUBTOKEN_SUPERVISION_POLICY,
        "partial_truncation_policy": PARTIAL_TRUNCATION_POLICY,
        "train_encoding": train_encoding_report,
        "validation_encoding": validation_encoding_report,
        **tokenizer_equivalence,
    },
    tokenizer=tokenizer_report,
    completed_epochs=completed_epochs,
    completed_optimizer_steps=global_step,
    validation_metrics=validation_metrics,
    checkpoint_hashes=checkpoint_hashes,
    run_completed=run_completed,
    interrupted_reason=interrupted_reason,
)
training_manifest["smoke_artifact"] = {
    "smoke_artifact_dir": str(SMOKE_ARTIFACT_DIR),
    "smoke_checkpoint_sha256": smoke_outcome.checkpoint_sha256,
    "smoke_artifact_validated": True,
    "used_as_initialization": False,
}
Path(OUTPUT_PATHS["training_manifest"]).write_text(
    json.dumps(training_manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({
    "status": FULL_TRAINING_MODE,
    "run_completed": run_completed,
    "interrupted_reason": interrupted_reason,
    "completed_epochs": completed_epochs,
    "completed_optimizer_steps": global_step,
    "best_metric_key": BEST_METRIC_KEY,
    "best_metric": best_metric,
    "checkpoint_sha256": checkpoint_hashes,
    "training_manifest_path": OUTPUT_PATHS["training_manifest"],
}, indent=2, sort_keys=True))

## 19. Return artifacts

Return `s1_mention_full_training_v1/` from Drive for review: `checkpoints/latest.pt`,
`checkpoints/best.pt`, `logs/training_history.jsonl`, `resolved_config.json`,
`validation_metrics.json`, and `training_manifest.json`.

The smoke artifact directory is untouched. No base-model cache lives in the training output,
no organizer inference was run, and no `output.zip` was produced.